# RoadGuard AI V1.1 — MobileNetV3-Small Scene Classifier Training Pipeline

**Mission**: Train a lightweight binary scene classifier (`0: ROAD`, `1: NON_ROAD`) to pre-filter camera frames.
**Model**: `MobileNetV3-Small` (ImageNet pretrained weights)
**Target Runtime**: Android ARM64 / ONNX Runtime INT8
**Hardware Acceleration**: Google Colab GPU (Tesla T4 / A100)

---

In [ ]:
# Cell 1: Environment Setup & High-Performance Dependencies
!nvidia-smi
!pip install --quiet torch torchvision onnx onnxruntime onnxslim scikit-learn matplotlib seaborn

In [ ]:
# Cell 2: Imports & Deterministic Seeding
import os, sys, time, json, random, shutil
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision
from torchvision import datasets, transforms, models

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

set_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Active Compute Device: {device}')

In [ ]:
# Cell 3: Google Drive Mounting & Persistence Directory Setup
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_OUTPUT_DIR = Path('/content/drive/MyDrive/RoadGuard_AI/V1_1_SceneClassifier')
except Exception as e:
    print(f'Running in local/offline environment: {e}')
    DRIVE_OUTPUT_DIR = Path('/content/output_scene_classifier')

DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f'Artifact persistence directory: {DRIVE_OUTPUT_DIR}')

In [ ]:
# Cell 4: Dataset Pipeline & Preprocessing Transformations
IMG_SIZE = 224
BATCH_SIZE = 64

train_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.25, contrast=0.25, saturation=0.20),
    transforms.RandomAffine(degrees=10, translate=(0.05, 0.05), scale=(0.95, 1.05)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_test_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

DATASET_ROOT = Path('/content/scene_classifier_dataset')
print(f'Configured input image size: {IMG_SIZE}x{IMG_SIZE}, batch: {BATCH_SIZE}')

In [ ]:
# Cell 5: MobileNetV3-Small Initialization & Transfer Learning Head
def build_mobilenet_v3_small(num_classes=2, pretrained=True):
    weights = models.MobileNet_V3_Small_Weights.DEFAULT if pretrained else None
    model = models.mobilenet_v3_small(weights=weights)
    
    # Replace top linear classifier
    in_features = model.classifier[3].in_features
    model.classifier[3] = nn.Sequential(
        nn.Dropout(p=0.2, inplace=True),
        nn.Linear(in_features, num_classes)
    )
    return model

model = build_mobilenet_v3_small(num_classes=2, pretrained=True).to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f'MobileNetV3-Small initialized. Total parameters: {total_params:,} ({total_params/1e6:.2f}M)')

In [ ]:
# Cell 6: Transfer Learning Training Loop with Early Stopping
EPOCHS = 20
LEARNING_RATE = 3e-4
PATIENCE = 5

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

best_val_loss = float('inf')
patience_counter = 0
history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

print('Ready to train when dataset is mounted in DATASET_ROOT.')

In [ ]:
# Cell 7: ONNX Export (Opset 17) & Dynamic INT8 Quantization
import onnx
import onnxruntime as ort
from onnxruntime.quantization import quantize_dynamic, QuantType

def export_and_quantize(model, output_dir=DRIVE_OUTPUT_DIR):
    model.eval()
    dummy_input = torch.randn(1, 3, 224, 224, device=device)
    fp32_path = output_dir / 'roadguard_scene_classifier_v1.1.0_fp32.onnx'
    int8_path = output_dir / 'roadguard_scene_classifier_v1.1.0_int8.onnx'
    
    # 1. Export FP32 ONNX
    torch.onnx.export(
        model,
        dummy_input,
        str(fp32_path),
        export_params=True,
        opset_version=17,
        do_constant_folding=True,
        input_names=['input'],
        output_names=['logits'],
        dynamic_axes={'input': {0: 'batch_size'}, 'logits': {0: 'batch_size'}}
    )
    print(f'Exported FP32 ONNX: {fp32_path}')
    
    # 2. Dynamic INT8 Quantization
    quantize_dynamic(
        model_input=str(fp32_path),
        model_output=str(int8_path),
        weight_type=QuantType.QUInt8
    )
    print(f'Exported INT8 ONNX: {int8_path}')
    return fp32_path, int8_path

In [ ]:
# Cell 8: On-Device Latency Benchmarking Protocol
def benchmark_onnx_latency(onnx_path, runs=100, warmup=20):
    session = ort.InferenceSession(str(onnx_path), providers=['CPUExecutionProvider'])
    dummy_data = np.random.randn(1, 3, 224, 224).astype(np.float32)
    
    for _ in range(warmup):
        _ = session.run(None, {'input': dummy_data})
        
    latencies = []
    for _ in range(runs):
        t0 = time.perf_counter()
        _ = session.run(None, {'input': dummy_data})
        latencies.append((time.perf_counter() - t0) * 1000.0)
        
    p50 = float(np.percentile(latencies, 50))
    p95 = float(np.percentile(latencies, 95))
    print(f'Latency ({onnx_path.name}): p50={p50:.2f}ms, p95={p95:.2f}ms (x86 CPU Proxy)')
    return p50, p95